# MMLU figures

Each base LLM answers every question once. Our aggregator groups identical answers and
returns the answer of the largest group if at least $\tau$ models support it; otherwise it
abstains. `VPA{τ}` is the aggregator's answer at plurality threshold $\tau$.

For each dataset this notebook produces two figures, written to `figures/`:

1. **Outcome breakdown**: share of questions answered correctly, answered incorrectly, and abstained on, for each $\tau$
   (`{name}_VPA_varying_tau_abs_error_acc.pdf`).
2. **Accuracy vs. $\tau$**: accuracy on the questions our aggregator answers, compared with each base model
   on the same questions (`Accuracy_vs_plurality_on_{name}.pdf`).

For MMLU-1k-SC, the outcome breakdown is also shown with one Byzantine voter added.

| Dataset | File | Questions | Models | $\tau$ |
|---|---|---|---|---|
| MMLU-PRO | `data/pro_mmlu_results.csv` | 200 | 12 | 2–6 |
| MMLU-1k-SC | `data/sc_mmlu_results.csv` | 1,000 | 7 | 3–6 |
| MMLU-300 | `data/mmlu_300qst_results.csv` | 300 | 10 | 4–7 |



In [ ]:
import numpy as np
import pandas as pd

from utils import (create_aggr_df, add_mimic, accuracy_vs_tau,
                   plot_acc_err_abs_mine, plot_accuracies_vs_coverage)

ID_COL, LABEL_COL = "question_id", "correct_label"
CHOICES_4  = np.array(list("ABCD"))
CHOICES_10 = np.array(list("ABCDEFGHIJ"))

## MMLU-PRO

200 questions with 10 answer options, 12 models.

In [ ]:
pro = pd.read_csv("data/pro_mmlu_results.csv")
pro_models = [c for c in pro.columns if c not in (ID_COL, LABEL_COL)]
PRO_TAUS = [2, 3, 4, 5, 6]

pro_res = create_aggr_df(pro, PRO_TAUS, ID_COL, LABEL_COL, CHOICES_10)

In [ ]:
plot_acc_err_abs_mine(pro_res, [f"VPA{k}" for k in PRO_TAUS], "MMLU-PRO")

In [ ]:
plot_accuracies_vs_coverage(accuracy_vs_tau(pro_res, PRO_TAUS, pro_models), "MMLU-PRO")

## MMLU-1k-SC

1,000 MMLU questions with 4 answer options, 7 models; each model's answer is its
self-consistency answer.

In [ ]:
# same model names as in the other datasets
SC_NAMES = {"gpt-120b": "gpt-oss-120b", "gpt-20b": "gpt-oss-20b",
            "llama70b": "llama-3.3-70b", "llama8b": "llama-3.1-8b"}
sc = pd.read_csv("data/sc_mmlu_results.csv").rename(columns=SC_NAMES)
sc_models = [c for c in sc.columns if c not in (ID_COL, LABEL_COL)]
SC_TAUS = [3, 4, 5, 6]

sc_res = create_aggr_df(sc, SC_TAUS, ID_COL, LABEL_COL, CHOICES_4)

In [ ]:
plot_acc_err_abs_mine(sc_res, [f"VPA{k}" for k in SC_TAUS], "MMLU-SC")

In [ ]:
plot_accuracies_vs_coverage(accuracy_vs_tau(sc_res, SC_TAUS, sc_models), "MMLU-SC")

### MMLU-1k-SC with one Byzantine voter

One extra voter is added to the 7 models:

- **mimic**: always gives the most common *wrong* answer among the models (ties broken at random).
- **random**: copies the answer of a randomly chosen model.

Both use seed 0.

In [ ]:
attacked  = add_mimic(sc, random_pick=True)          # adds the "mimic" and "random_pick" voters
sc_mimic  = attacked.drop(columns="random_pick")
sc_random = attacked.drop(columns="mimic")

mimic_res  = create_aggr_df(sc_mimic,  SC_TAUS, ID_COL, LABEL_COL, CHOICES_4)
random_res = create_aggr_df(sc_random, SC_TAUS, ID_COL, LABEL_COL, CHOICES_4)

In [ ]:
plot_acc_err_abs_mine(mimic_res, [f"VPA{k}" for k in SC_TAUS], "MMLU-SC-mimic")

In [ ]:
plot_acc_err_abs_mine(random_res, [f"VPA{k}" for k in SC_TAUS], "MMLU-SC-random")

## MMLU-300

The first 300 MMLU questions with 4 answer options, 10 models.

In [ ]:
m300 = pd.read_csv("data/mmlu_300qst_results.csv").drop(columns="qwen3.6-27b")  # not in the 10-model pool
m300_models = [c for c in m300.columns if c not in (ID_COL, LABEL_COL)]
M300_TAUS = [4, 5, 6, 7]

m300_res = create_aggr_df(m300, M300_TAUS, ID_COL, LABEL_COL, CHOICES_4)

In [ ]:
plot_acc_err_abs_mine(m300_res, [f"VPA{k}" for k in M300_TAUS], "MMLU-300")

In [ ]:
plot_accuracies_vs_coverage(accuracy_vs_tau(m300_res, M300_TAUS, m300_models), "MMLU-300")